# Chapter 9 — Activation Functions

Part 3: Small mathematical operations.

**Ladder:** scalar Python activations (Method 01) → one Triton elementwise kernel with a compile-time `MODE` (Method 02) → fused bias + activation and gated epilogues (Method 03).

**Study tip.** Take

$$
x=(-2,\,-1,\,0,\,1,\,2)
$$

and compute ReLU, sigmoid, tanh, GELU, SiLU, hard sigmoid, hard swish, and softplus by hand to three decimals. Then write each derivative from the table in Part I. The numerical cells below are the answer key, not the first exposure. Pay particular attention to the two boundaries: ReLU at $0$ and the hard activations at $\pm3$.

**Learning objectives**

- Read an activation as an elementwise lift that preserves the input shape.
- State the **same-shape rule** and the **broadcast rule** for a fused bias + activation.
- Implement the ten book activations in one Triton kernel selected by a `tl.constexpr`.
- Choose a numerically stable form for sigmoid, tanh, GELU, and softplus.
- Derive and check the backward pass of every activation in the table.
- Fuse a bias add into the activation without materializing the intermediate.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch08_comparisons/08_complete.ipynb) · [Next →](../../part4_broadcasting/ch10_broadcasting/10_complete.ipynb)

**Estimated time:** 65–80 min · **Difficulty:** Intermediate

**Prerequisites:** Chapter 7 (exp, log, maximum, clamp) and Chapter 8 (`tl.where`, masks).

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Chapter map

| § | Activation | Key formula | Shape rule | Applied kernel |
|---|---|---|---|---|
| **9.1** | ReLU | $\max(x,0)$ | same shape | ReLU / leaky ReLU |
| **9.2** | Leaky ReLU | $\max(x,0)+\alpha\min(x,0)$ | same shape | ReLU / leaky ReLU |
| **9.3** | Sigmoid | $1/(1+e^{-x})$ | same shape | sigmoid / tanh |
| **9.4** | Tanh | $\tanh x$ | same shape | sigmoid / tanh |
| **9.5** | GELU | $\frac{x}{2}(1+\operatorname{erf}(x/\sqrt2))$ | same shape | modern activations |
| **9.6** | SiLU / Swish | $x\,\sigma(x)$ | same shape | modern activations |
| **9.7** | Hard sigmoid | $\min(\max((x+3)/6,0),1)$ | same shape | modern activations |
| **9.8** | Hard swish | $x\,\mathrm{hard\_sigmoid}(x)$ | same shape | modern activations |
| **9.9** | Softplus | $\max(x,0)+\ln(1+e^{-\lvert x\rvert})$ | same shape | softplus + threshold |
| **9.10** | Clamp / threshold | $\min(\max(x,l),u)$ | same shape | softplus + threshold |
| **9.11** | Runnable activation | $y_i=\operatorname{act}(x_i)$ | same shape | compile-time selector |
| **★** | Capstone | $y=\operatorname{act}(x+b)$ | bias broadcasts on the last axis | fused MLP epilogue |

**One-line thesis.** An activation is a scalar nonlinearity lifted to a tile. The
forward and the backward are both elementwise; the only interesting decisions are
numerical stability, the boundary convention, and whether the bias add is fused.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import math

import pytest
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

## Concept diagrams

### Elementwise activation ladder

```text
Method 01                 Method 02                    Method 03
scalar Python loop   ->   one Triton tile kernel   ->   fused bias + activation
one value at a time       load f(x) store               load (x, b), y=act(x+b)

slow, explicit            parallel, masked              one pass, no intermediate
the reference              the workhorse                 the production epilogue
```

### The tile does not change shape

```text
x:      [ x0  x1  x2  x3 | x4  x5  x6  * ]
mask:   [  1   1   1   1 |  1   1   1  0 ]
act:    [ f0  f1  f2  f3 | f4  f5  f6  * ]
y:      [ y0  y1  y2  y3 | y4  y5  y6  * ]   (the last slot is not stored)
```

### Broadcast in a fused epilogue

```text
X:      (M, N)        bias: (N,)        out: (M, N)
        +------------------+
        | x[m,n] + b[n]    |  -> act -> y[m,n]
        +------------------+
```

The bias has one value per column. It is logically replicated over the $M$ rows;
the kernel can read it once per column tile and add it to every row.

# Part I — Mathematics

**Thread of the chapter.** An activation is a *gate*: it keeps a value, scales
it, or suppresses it, lane by lane. The gate is elementwise, so the shape never
changes. The chapter follows one chain:

- **ReLU** is the simplest gate: keep the positive part.
- **Leaky ReLU** adds a small negative slope.
- **Sigmoid** makes the gate smooth and bounded.
- **Tanh** is a shifted, scaled sigmoid.
- **GELU** and **SiLU** multiply a value by a smooth gate.
- **Hard sigmoid** and **hard swish** replace the smooth gate by a piecewise
  linear one.
- **Softplus** is the smooth positive part; **clamp** is the safety net.

Every section below says which earlier link it extends.

## 1. The activation contract

**Connection.** Chapter 7 established the elementwise lift; Chapter 8 supplied
`tl.where` and masks. An activation is the elementwise lift with a nonlinear
$f$.

- **Shape rule.** $y_i=f(x_i)$ and $\operatorname{shape}(y)=\operatorname{shape}(x)$.
- **Backward.** $dx_i=g_i f'(x_i)$; the backward is another elementwise kernel.
- **Storage vs compute.** Load narrow, compute in `float32`, cast once at the
  store. `exp`, `log`, `erf`, and `tanh` are defined for `float32`/`float64`.
- **Kinks are API.** ReLU at $0$, hard sigmoid at $\pm3$, clamp at $l,u$: the
  derivative is not unique. The convention must be stated and tested.
- **Bridge to §2.** The zoo below is the list of $f$'s; each row is one link in
  the chain above.

## 2. The activation zoo

**Connection.** Each row extends an earlier row: ReLU $\to$ leaky ReLU $\to$
sigmoid $\to$ tanh $\to$ GELU/SiLU $\to$ hard variants $\to$ softplus/clamp.

| § | Name | Formula | Range | Derivative | Extends |
|---|---|---|---|---|---|
| **9.1** | ReLU | $\max(x,0)$ | $[0,\infty)$ | $[x\gt 0]$ | Chapter 7 `maximum` |
| **9.2** | Leaky ReLU | $\max(x,0)+\alpha\min(x,0)$ | $\mathbb R$ | $1$ or $\alpha$ | ReLU |
| **9.3** | Sigmoid | $1/(1+e^{-x})$ | $(0,1)$ | $\sigma(1-\sigma)$ | Chapter 7 `exp`, division |
| **9.4** | Tanh | $\tanh x$ | $(-1,1)$ | $1-\tanh^2 x$ | sigmoid: $2\sigma(2x)-1$ |
| **9.5** | GELU | $\frac{x}{2}(1+\operatorname{erf}(x/\sqrt2))$ | $(\approx-0.17,\infty)$ | $\Phi(x)+x\phi(x)$ | sigmoid/tanh gate |
| **9.6** | SiLU | $x\,\sigma(x)$ | $(\approx-0.28,\infty)$ | $\sigma(1+x(1-\sigma))$ | sigmoid |
| **9.7** | Hard sigmoid | $\min(\max((x+3)/6,0),1)$ | $[0,1]$ | $1/6$ on $(-3,3)$ | clamp |
| **9.8** | Hard swish | $x\,\mathrm{hs}(x)$ | $(\approx-0.375,\infty)$ | piecewise | hard sigmoid |
| **9.9** | Softplus | $\ln(1+e^x)$ | $(0,\infty)$ | $\sigma(x)$ | ReLU + `exp`/`log` |
| **9.10** | Clamp | $\min(\max(x,l),u)$ | $[l,u]$ | $1$ on $(l,u)$ | Chapter 7 `clamp` |

- **Smooth vs hard.** Rows 9.1–9.6 are smooth or piecewise linear; 9.7–9.8 are
  the cheap piecewise-linear approximations; 9.9 is the smooth positive part.
- **Gates.** Sigmoid, tanh, GELU, SiLU, hard sigmoid, and hard swish all produce
  a gate in $(0,1)$; the value is either multiplied by the gate (SiLU) or by a
  normalized CDF (GELU).
- **Boundary conventions.**
  - ReLU: $1$ for $x\gt 0$, $0$ for $x\le0$.
  - Leaky ReLU: $1$ for $x\ge0$, $\alpha$ otherwise.
  - Hard sigmoid: $1/6$ on $(-3,3)$, $0$ outside.
  - Hard swish: $0$, $(2x+3)/6$, $1$ on the three pieces.
  - Clamp: $1$ on $(l,u)$, $0$ outside.
- **Bridge to §3.** The formulas are exact; the implementation must survive
  `float32` overflow. The next section chooses the stable forms.

## 3. Numerical stability

**Connection.** Chapter 7 warned that `exp` overflows near $88.7$ in `float32`.
Each activation below inherits that warning in a different algebraic form.

### Sigmoid

- **Stable form.** $1/(1+e^{-x})$: for very negative $x$, $e^{-x}$ overflows to
  $+\infty$ and the reciprocal is $0$, which is the correct limit.
- **Unstable form.** $e^x/(1+e^x)$ computes $\infty/\infty=\text{nan}$ for large
  positive $x$.
- **Rule.** Use `tl.sigmoid(x)`, or the first form, never the second.

### Tanh

- **Stable form.** $\tanh x=2\sigma(2x)-1$ reuses the stable sigmoid.
- **Unstable form.** The quotient $(e^x-e^{-x})/(e^x+e^{-x})$ overflows for
  $\lvert x\rvert\gtrsim44$.
- **Rule.** Use `libdevice.tanh` when available, or the identity.

### GELU

- **Exact form.** $\frac{x}{2}(1+\operatorname{erf}(x/\sqrt2))$.
- **Tanh approximation.**
  $\frac{x}{2}(1+\tanh(\sqrt{2/\pi}(x+0.044715x^3)))$.
- **Accuracy.** About $10^{-3}$ on $[-5,5]$. A test must use a tolerance that
  reflects the approximation, not `1e-6`.
- **Rule.** The exact form is the reference; the approximation is a different
  function.

### Softplus

- **Stable form.** $\max(x,0)+\log(1+e^{-\lvert x\rvert})$.
- **Why.** For $x\gt 0$ the first term carries the large value; for $x\lt 0$ the
  exponential is at most one. The result is finite for every finite `float32`.
- **Unstable form.** $\log(1+e^x)$ overflows for $x\gtrsim88.7$.
- **Bridge to §4.** A stable forward is only half the story; the backward must
  also be finite and must match the local derivative.

In [ ]:
x = torch.tensor([-100.0, -10.0, 0.0, 10.0, 100.0], dtype=torch.float32)
assert x.shape == (5,)

stable_sig = torch.sigmoid(x)
naive_sig = torch.exp(x) / (1.0 + torch.exp(x))
assert torch.isnan(naive_sig[-1]) and not torch.isnan(stable_sig).any()
assert torch.allclose(stable_sig, 1.0 / (1.0 + torch.exp(-x)))

big = torch.tensor([80.0, 88.0, 89.0, 100.0])
naive_sp = torch.log(1.0 + torch.exp(big))
stable_sp = torch.maximum(big, torch.zeros_like(big)) + torch.log1p(torch.exp(-big.abs()))
assert torch.isinf(naive_sp[2:]).all() and torch.isfinite(stable_sp).all()
assert torch.allclose(stable_sp, F.softplus(big), atol=1e-5, rtol=1e-5)
print("stability: the stable forms stay finite where the naive forms overflow")

## 4. Derivatives and the backward pass

**Connection.** §3 stabilized the forward. This section differentiates it; the
backward is another elementwise kernel with the same shape and mask.

**Chain rule.** For $y=f(x)$ and upstream gradient $g$, $dx_i=g_i f'(x_i)$.

- **GELU.** With $u=x/\sqrt2$,
  $f'(x)=\frac12(1+\operatorname{erf}u)+\frac{x}{\sqrt{2\pi}}e^{-x^2/2}$.
  At $x=0$ this is $1/2$.
- **SiLU.** With $s=\sigma(x)$,
  $f'(x)=s+x s(1-s)=s(1+x(1-s))$.
- **Hard swish.** On $-3\lt x\lt 3$, $f(x)=x(x+3)/6$ and $f'(x)=(2x+3)/6$; outside,
  $0$ on the left and $1$ on the right.
- **Softplus.** $f'(x)=\sigma(x)$. This is the cleanest check in the chapter.
- **Why it matters.** The backward of a fused forward is another elementwise
  pass. The mask/condition is the only state that must be saved (or recomputed).
- **Bridge to §5.** The next section puts the forward values and derivatives on
  one tiny spine so every formula can be checked by hand.

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
assert x.shape == (5,)

def manual_derivatives(x):
    s = torch.sigmoid(x)
    t = torch.tanh(x)
    u = x / math.sqrt(2.0)
    gelu = 0.5 * (1.0 + torch.erf(u)) + x / math.sqrt(2.0 * math.pi) * torch.exp(-x * x / 2.0)
    hsig = ((x > -3.0) & (x < 3.0)).float() / 6.0
    hswish = torch.where(x < -3.0, torch.zeros_like(x),
                         torch.where(x > 3.0, torch.ones_like(x), (2.0 * x + 3.0) / 6.0))
    return {
        "relu": (x > 0).float(),
        "sigmoid": s * (1.0 - s),
        "tanh": 1.0 - t * t,
        "gelu": gelu,
        "silu": s * (1.0 + x * (1.0 - s)),
        "hardsigmoid": hsig,
        "hardswish": hswish,
        "softplus": s,
    }

expected = {
    "relu": [0.0, 0.0, 0.0, 1.0, 1.0],
    "sigmoid": [0.10499, 0.19661, 0.25, 0.19661, 0.10499],
    "tanh": [0.07065, 0.41997, 1.0, 0.41997, 0.07065],
    "gelu": [-0.08523, -0.08332, 0.5, 1.08332, 1.08523],
    "silu": [-0.09078, 0.07233, 0.5, 0.92767, 1.09078],
    "hardsigmoid": [1 / 6] * 5,
    "hardswish": [-0.16667, 0.16667, 0.5, 0.83333, 1.16667],
    "softplus": [0.11920, 0.26894, 0.5, 0.73106, 0.88080],
}
for name, value in manual_derivatives(x).items():
    assert torch.allclose(value, torch.tensor(expected[name]), atol=1e-4), name
print("derivative table matches the text")

## 5. Worked numerical example — the chapter spine

**Connection.** This is the single table every earlier section feeds into:
each column is one activation from §9.1–9.9, evaluated at the same five points.

Take $x=(-2,-1,0,1,2)$. The forward values, to three decimals, are

| $x$ | ReLU | Sigmoid | Tanh | GELU | SiLU | Hard sigmoid | Hard swish | Softplus |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| -2 | 0.000 | 0.119 | -0.964 | -0.046 | -0.238 | 0.167 | -0.333 | 0.127 |
| -1 | 0.000 | 0.269 | -0.762 | -0.159 | -0.269 | 0.333 | -0.333 | 0.313 |
| 0 | 0.000 | 0.500 | 0.000 | 0.000 | 0.000 | 0.500 | 0.000 | 0.693 |
| 1 | 1.000 | 0.731 | 0.762 | 0.841 | 0.731 | 0.667 | 0.667 | 1.313 |
| 2 | 2.000 | 0.881 | 0.964 | 1.954 | 1.762 | 0.833 | 1.667 | 2.127 |

**Hand checks.**

- $\sigma(0)=1/2$ and $\tanh(0)=0$.
- $\operatorname{GELU}(0)=0$; at $x=1$ it is $\Phi(1)\approx0.841$.
- $\operatorname{SiLU}(1)=\sigma(1)\approx0.731$.
- Hard swish at $x=-1$ is $(-1)(2)/6=-1/3$; at $x=1$ it is $4/6=2/3$.
- $\operatorname{softplus}(0)=\log2\approx0.693$.
- The derivative table in §4 is checked against autograd in the code cell after
  the table.

**Bridge to §6.** GELU is the one row with two implementations; the next section
quantifies their difference.

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
u = x / math.sqrt(2.0)
gelu = 0.5 * x * (1.0 + torch.erf(u))
hsig = torch.clamp((x + 3.0) / 6.0, 0.0, 1.0)
hswish = x * hsig
softplus = torch.maximum(x, torch.zeros_like(x)) + torch.log1p(torch.exp(-x.abs()))

table = {
    "relu": torch.relu(x),
    "sigmoid": torch.sigmoid(x),
    "tanh": torch.tanh(x),
    "gelu": gelu,
    "silu": x * torch.sigmoid(x),
    "hardsigmoid": hsig,
    "hardswish": hswish,
    "softplus": softplus,
}
assert all(v.shape == (5,) for v in table.values())
expected = {
    "relu": [0.0, 0.0, 0.0, 1.0, 2.0],
    "sigmoid": [0.119, 0.269, 0.500, 0.731, 0.881],
    "tanh": [-0.964, -0.762, 0.000, 0.762, 0.964],
    "gelu": [-0.045, -0.159, 0.000, 0.841, 1.955],
    "silu": [-0.238, -0.269, 0.000, 0.731, 1.762],
    "hardsigmoid": [0.167, 0.333, 0.500, 0.667, 0.833],
    "hardswish": [-0.333, -0.333, 0.000, 0.667, 1.667],
    "softplus": [0.127, 0.313, 0.693, 1.313, 2.127],
}
for name, want in expected.items():
    got = table[name]
    assert torch.allclose(got, torch.tensor(want), atol=1e-3), (name, got.tolist(), want)
print("worked table: all entries match the notes")

## 6. GELU: exact versus tanh approximation

**Connection.** §9.5 introduced GELU as a smooth gate built from `erf`. This
section compares that exact form with the tanh approximation used in inference.

- **Exact.** $\operatorname{GELU}(x)=\frac{x}{2}(1+\operatorname{erf}(x/\sqrt2))$.
- **Approximation.**
  $\frac{x}{2}(1+\tanh(\sqrt{2/\pi}(x+0.044715x^3)))$.
- **Accuracy.** Maximum absolute error is about $4.7\times10^{-4}$ on $[-5,5]$;
  the relative error near zero is not meaningful because both values are small.
- **Rule.** Compare the approximation to the exact form with a stated tolerance
  ($6\times10^{-4}$ in the test), never `1e-6`.
- **Why it exists.** Some inference stacks have a cheap `tanh` already in the
  fused kernel; the exact form is the training reference.
- **Bridge to §7.** Every activation in the table can be fused with a bias; the
  next section states the broadcast rule for that epilogue.

In [ ]:
x = torch.linspace(-5.0, 5.0, 2001)
exact = F.gelu(x)
approx = 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)))

assert x.shape == exact.shape == approx.shape == (2001,)
assert (exact - approx).abs().max().item() < 6e-4

spine = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
spine_approx = 0.5 * spine * (1.0 + torch.tanh(
    math.sqrt(2.0 / math.pi) * (spine + 0.044715 * spine**3)))
assert torch.allclose(spine_approx, F.gelu(spine), atol=6e-4)
print("GELU: the tanh approximation is within 6e-4 on [-5, 5]")

## 7. Broadcasting a bias into an activation

**Connection.** Chapters 8 and 10 established the broadcast rule. The fused
epilogue is its most common use: a per-column bias added to a matrix before the
activation.

- **Formula.** $Y_{mn}=f(X_{mn}+b_n)$ for $X\in\mathbb R^{M\times N}$ and
  $b\in\mathbb R^{N}$.
- **Shape rule.** $\operatorname{shape}(X+b)=\operatorname{broadcast\_shapes}((M,N),(N,))=(M,N)$.
- **What breaks.** A bias of shape $(M,)$ does not broadcast over the columns of
  $(M,N)$; the last axes are $N$ and $M$. Use $(N,)$ or reshape to $(M,1)$.
- **Kernel rule.** Load one bias value per column tile and add it to every row:
  `b = tl.load(bias_ptr + cols); z = x + b[None, :]`.
- **No materialization.** The expanded bias is a view with a zero stride; the
  fused kernel never writes the $(M,N)$ sum to HBM.
- **Bridge to Part II.** The next part writes the scalar reference for all ten
  activations, then Part III implements them in one tile kernel.

In [ ]:
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
bias = torch.tensor([10.0, 20.0, 30.0])

assert X.shape == (2, 3) and bias.shape == (3,)
assert torch.broadcast_shapes(X.shape, bias.shape) == (2, 3)
assert (X + bias).tolist() == [[11.0, 22.0, 33.0], [14.0, 25.0, 36.0]]

col_bias = torch.tensor([[10.0], [20.0]])
assert torch.broadcast_shapes(X.shape, col_bias.shape) == (2, 3)
assert (X + col_bias).tolist() == [[11.0, 12.0, 13.0], [24.0, 25.0, 26.0]]

try:
    torch.broadcast_shapes((2, 3), (2,))
except RuntimeError:
    pass
else:
    raise AssertionError("(2,3) and (2,) should not broadcast")
print("bias broadcast: the last-axis rule and the shape error match the text")

# Part II — Method 01: scalar activations

The reference implementation is a Python loop over elements. It is the
specification: one value in, one value out, no vectorization and no masks. The
loop makes the boundary conventions explicit because each branch is visible.

Ten functions are enough, but a single dispatch table keeps the reference short.
The table is also the answer key for the Method 02 selector: the same `MODE`
integers are used in both implementations.

In [ ]:
MODE_RELU, MODE_LEAKY, MODE_SIGMOID, MODE_TANH, MODE_GELU, \
MODE_SILU, MODE_HSIG, MODE_HSWISH, MODE_SOFTPLUS, MODE_CLAMP = range(10)

MODE_NAMES = [
    "relu", "leaky_relu", "sigmoid", "tanh", "gelu",
    "silu", "hardsigmoid", "hardswish", "softplus", "clamp",
]


def scalar_activation(x: float, mode: int, alpha: float = 0.01,
                      lo: float = 0.0, hi: float = 1.0) -> float:
    '''Method 01: one scalar activation, with the boundary conventions stated.'''
    if mode == MODE_RELU:
        return x if x > 0.0 else 0.0
    if mode == MODE_LEAKY:
        return x if x >= 0.0 else alpha * x
    if mode == MODE_SIGMOID:
        return 1.0 / (1.0 + math.exp(-x))
    if mode == MODE_TANH:
        return math.tanh(x)
    if mode == MODE_GELU:
        return 0.5 * x * (1.0 + math.erf(x / math.sqrt(2.0)))
    if mode == MODE_SILU:
        return x / (1.0 + math.exp(-x))
    if mode == MODE_HSIG:
        return min(max((x + 3.0) / 6.0, 0.0), 1.0)
    if mode == MODE_HSWISH:
        return x * min(max((x + 3.0) / 6.0, 0.0), 1.0)
    if mode == MODE_SOFTPLUS:
        return max(x, 0.0) + math.log1p(math.exp(-abs(x)))
    if mode == MODE_CLAMP:
        return min(max(x, lo), hi)
    raise ValueError(mode)


def naive_activation(x: torch.Tensor, mode: int, **kwargs) -> torch.Tensor:
    '''Method 01: apply the scalar function element by element.'''
    out = torch.empty_like(x, dtype=torch.float32)
    xf, of = x.reshape(-1), out.reshape(-1)
    for i in range(xf.numel()):
        of[i] = scalar_activation(float(xf[i]), mode, **kwargs)
    return out.reshape(x.shape)


x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("x shape:", tuple(x.shape))
for mode, name in enumerate(MODE_NAMES):
    y = naive_activation(x, mode)
    print(f"{name:>11}: shape {tuple(y.shape)} | {[round(v, 3) for v in y.tolist()]}")

In [ ]:
def test_worked_example() -> None:
    '''Check the exact numbers in the Part I table.'''
    x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
    expected = {
        "relu": [0.0, 0.0, 0.0, 1.0, 2.0],
        "leaky_relu": [-0.02, -0.01, 0.0, 1.0, 2.0],
        "sigmoid": [0.119, 0.269, 0.500, 0.731, 0.881],
        "tanh": [-0.964, -0.762, 0.000, 0.762, 0.964],
        "gelu": [-0.045, -0.159, 0.000, 0.841, 1.955],
        "silu": [-0.238, -0.269, 0.000, 0.731, 1.762],
        "hardsigmoid": [0.167, 0.333, 0.500, 0.667, 0.833],
        "hardswish": [-0.333, -0.333, 0.000, 0.667, 1.667],
        "softplus": [0.127, 0.313, 0.693, 1.313, 2.127],
        "clamp": [0.0, 0.0, 0.0, 1.0, 1.0],
    }
    refs = [
        torch.relu(x),
        torch.nn.functional.leaky_relu(x, 0.01),
        torch.sigmoid(x),
        torch.tanh(x),
        torch.nn.functional.gelu(x),
        torch.nn.functional.silu(x),
        torch.nn.functional.hardsigmoid(x),
        torch.nn.functional.hardswish(x),
        torch.nn.functional.softplus(x),
        torch.clamp(x, 0.0, 1.0),
    ]
    for mode, name in enumerate(MODE_NAMES):
        ref = refs[mode]
        got = naive_activation(x, mode)
        assert torch.allclose(got, torch.tensor(expected[name], dtype=torch.float32), atol=1e-3)
        assert torch.allclose(got, ref, atol=1e-6), name
    print("worked example: scalar activations match the table and PyTorch")


def test_naive_reference() -> None:
    '''The scalar reference agrees with PyTorch on a random vector.'''
    torch.manual_seed(0)
    x = torch.randn(50) * 2.0
    refs = [
        torch.relu(x),
        torch.nn.functional.leaky_relu(x, 0.01),
        torch.sigmoid(x),
        torch.tanh(x),
        torch.nn.functional.gelu(x),
        torch.nn.functional.silu(x),
        torch.nn.functional.hardsigmoid(x),
        torch.nn.functional.hardswish(x),
        torch.nn.functional.softplus(x),
        torch.clamp(x, 0.0, 1.0),
    ]
    for mode, ref in enumerate(refs):
        got = naive_activation(x, mode)
        assert torch.allclose(got, ref, atol=2e-2, rtol=2e-2), MODE_NAMES[mode]
    print("method 01: all ten activations match PyTorch")


test_worked_example()
test_naive_reference()

# Part III — Method 02: the Triton activation kernel

## 8. One kernel, ten modes

All ten activations share the tile skeleton

$$
\text{offs}=\text{pid}\cdot B+\{0,\ldots,B-1\},
\qquad
\text{mask}=\text{offs}\lt N,
\qquad
P=\left\lceil N/B\right\rceil.
$$

The mode is a `tl.constexpr`, so the JIT compiles a separate specialized body for
each activation. There is no runtime switch and no branch divergence.

**Tile map for $N=7$, $B=4$.** $P=2$; the last lane of the second program is
masked.

| `pid` | `offs` | `mask` | existing lanes |
|---:|---|---|---|
| 0 | 0,1,2,3 | 1,1,1,1 | 0,1,2,3 |
| 1 | 4,5,6,7 | 1,1,1,0 | 4,5,6 |

**Dtype rule.** The kernel loads into `float32`, evaluates the activation in
`float32`, and stores back to the output pointer's dtype. In this chapter the
output is `float32`; a mixed-precision version would cast once at the end.

**Library calls.** `tl.erf` is the explicit error function. Tanh is written as
`2*sigmoid(2x)-1`, which reuses the stable logistic and avoids the quotient of
exponentials. `tl.clamp` and `tl.minimum`/`tl.maximum` are branch-free.

In [ ]:
@triton.jit
def m9_activation_kernel(x_ptr, out_ptr, n_elements,
                         MODE: tl.constexpr, ALPHA: tl.constexpr,
                         LO: tl.constexpr, HI: tl.constexpr,
                         BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0).to(tl.float32)

    if MODE == 0:          # ReLU
        y = tl.maximum(x, 0.0)
    elif MODE == 1:        # Leaky ReLU
        y = tl.where(x >= 0.0, x, ALPHA * x)
    elif MODE == 2:        # Sigmoid
        y = tl.sigmoid(x)
    elif MODE == 3:        # Tanh: 2*sigmoid(2x) - 1, no quotient overflow
        y = 2.0 * tl.sigmoid(2.0 * x) - 1.0
    elif MODE == 4:        # GELU, exact erf
        y = 0.5 * x * (1.0 + tl.erf(x * 0.7071067811865476))
    elif MODE == 5:        # SiLU / Swish
        y = x * tl.sigmoid(x)
    elif MODE == 6:        # Hard sigmoid
        y = tl.minimum(tl.maximum((x + 3.0) / 6.0, 0.0), 1.0)
    elif MODE == 7:        # Hard swish
        y = x * tl.minimum(tl.maximum(x + 3.0, 0.0), 6.0) / 6.0
    elif MODE == 8:        # Softplus, stable
        y = tl.maximum(x, 0.0) + tl.log(1.0 + tl.exp(-tl.abs(x)))
    else:                  # Clamp / threshold
        y = tl.clamp(x, LO, HI)

    tl.store(out_ptr + offs, y, mask=mask)


def activation(x, mode, BLOCK=256, alpha=0.01, lo=0.0, hi=1.0):
    '''Method 02: one tile kernel, one compile-time activation mode.'''
    x = torch.as_tensor(x).contiguous()
    if x.dtype != torch.float32:
        raise TypeError("this chapter's kernel expects float32 input")
    out = torch.empty_like(x)
    m9_activation_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(),
        MODE=mode, ALPHA=alpha, LO=lo, HI=hi, BLOCK=BLOCK,
    )
    return out


def test_activation_kernel():
    require_gpu("m9 activation kernel")
    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE) * 2.0
    refs = [
        torch.relu(x),
        torch.nn.functional.leaky_relu(x, 0.01),
        torch.sigmoid(x),
        torch.tanh(x),
        torch.nn.functional.gelu(x),
        torch.nn.functional.silu(x),
        torch.nn.functional.hardsigmoid(x),
        torch.nn.functional.hardswish(x),
        torch.nn.functional.softplus(x),
        torch.clamp(x, 0.0, 1.0),
    ]
    for mode, ref in enumerate(refs):
        check(f"activation {MODE_NAMES[mode]}", activation(x, mode), ref,
              atol=2e-5, rtol=2e-5)

    # The tail path: N=7, BLOCK=4.
    xs = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0], device=DEVICE)
    check("activation tail relu", activation(xs, MODE_RELU, BLOCK=4), torch.relu(xs))


run_if_gpu(test_activation_kernel)

## 9. Stability checks that the kernel must pass

The stability claims are testable. The next cell compares three forms:

1. the kernel's stable sigmoid against `torch.sigmoid`;
2. the kernel's stable softplus against `torch.nn.functional.softplus`;
3. the naive `log(1+exp(x))` softplus against the stable one on large inputs.

The kernel should be finite on every input in the test, including `x=100` and
`x=-100`. The naive form is expected to produce `inf`; that is the point of the
comparison.

**A note on `exp` and `log` types.** In Triton, `tl.exp` and `tl.log` are defined
for `float32` and `float64`. The kernel casts the loaded tile to `float32`
first, so a `float16` input would be promoted rather than passed to an
unsupported overload.

In [ ]:
def test_stability_forms():
    require_gpu("m9 stability")
    x = torch.tensor([-100.0, -10.0, 0.0, 10.0, 100.0], device=DEVICE)
    check("stable sigmoid at extremes", activation(x, MODE_SIGMOID), torch.sigmoid(x),
          atol=1e-6, rtol=1e-6)
    check("stable softplus at extremes", activation(x, MODE_SOFTPLUS),
          torch.nn.functional.softplus(x), atol=1e-5, rtol=1e-5)
    out = activation(x, MODE_SOFTPLUS)
    assert torch.isfinite(out).all(), out
    print("stable forms are finite on [-100, 100]")


run_if_gpu(test_stability_forms)

## 10. The boundary convention is part of the test

Piecewise activations have a derivative convention at the join. The forward is
unambiguous; the backward is not. The cell below prints the forward value and
the autograd gradient at $x=-3$, $x=0$, and $x=3$ for hard sigmoid, hard swish,
and ReLU. Read the printed values against the conventions in Part I.

**What to look for.**

- ReLU at zero: PyTorch's gradient is $0$ because the condition is `x>0`.
- Hard sigmoid at $-3$ and $3$: PyTorch's gradient is $0$ at both joins.
- Hard swish at $-3$ and $3$: PyTorch's gradient is $0$ at $-3$ and $1$ at $3$.

If a kernel uses `>=` where PyTorch uses `>`, the only difference is at the
boundary. A test that never feeds the boundary will not see it.

In [ ]:
points = torch.tensor([-3.0, 0.0, 3.0])
expected = {
    "relu": [0.0, 0.0, 1.0],
    "hardsigmoid": [0.0, 1 / 6, 0.0],
    "hardswish": [0.0, 0.5, 1.0],
    "clamp": [0.0, 1.0, 0.0],
}
fns = {
    "relu": torch.relu,
    "hardsigmoid": torch.nn.functional.hardsigmoid,
    "hardswish": torch.nn.functional.hardswish,
    "clamp": lambda t: torch.clamp(t, -1.0, 1.0),
}
for name, fn in fns.items():
    x = points.clone().requires_grad_(True)
    fn(x).sum().backward()
    assert torch.allclose(x.grad, torch.tensor(expected[name])), name
print("boundary gradients: the convention matches PyTorch at -3, 0, 3")

# Part IV — Method 03: fused bias + activation

## 11. The fused epilogue

A linear layer produces $X=AW^\top$ and then applies a bias and an activation:

$$
Y=f(X+b).
$$

The unfused version materializes $X+b$ in HBM, reads it back, and writes $Y$.
The fused version keeps the sum in registers and writes only $Y$. For a tile of
$M\times N$ `float32` values, the unfused version touches an extra $2MN\cdot4$
bytes (write then read) before the activation output.

**The 2-D kernel.** Program `(pid_m, pid_n)` owns the tile

$$
\text{rows}=\text{pid}_m B_M+\{0,\ldots,B_M-1\},
\qquad
\text{cols}=\text{pid}_n B_N+\{0,\ldots,B_N-1\}.
$$

The input address is `rows[:,None]*N + cols[None,:]`; the bias address is
`cols`; the output address is the same as the input. The bias is loaded once per
column tile and broadcast over the rows by the `[None, :]` shape.

**Mask rule.** Both `rows<M` and `cols<N` are needed. A 2-D mask is the outer
AND:

```python
mask = (rows < M)[:, None] & (cols < N)[None, :]
```

For a partial tile, both the load and the store use this mask. The bias load
uses only the column mask, because the bias has no row axis.

In [ ]:
MODE_RELU, MODE_LEAKY, MODE_SIGMOID, MODE_TANH, MODE_GELU, \
MODE_SILU, MODE_HSIG, MODE_HSWISH, MODE_SOFTPLUS, MODE_CLAMP = range(10)


@triton.jit
def m9_bias_activation_kernel(x_ptr, bias_ptr, out_ptr, M, N,
                              MODE: tl.constexpr, BLOCK_M: tl.constexpr,
                              BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    row_mask = rows < M
    col_mask = cols < N
    mask = row_mask[:, None] & col_mask[None, :]

    offsets = rows[:, None].to(tl.int64) * N + cols[None, :]
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
    b = tl.load(bias_ptr + cols, mask=col_mask, other=0.0).to(tl.float32)

    z = x + b[None, :]
    if MODE == MODE_RELU:
        y = tl.maximum(z, 0.0)
    elif MODE == MODE_SIGMOID:
        y = tl.sigmoid(z)
    elif MODE == MODE_TANH:
        y = 2.0 * tl.sigmoid(2.0 * z) - 1.0
    elif MODE == MODE_GELU:
        y = 0.5 * z * (1.0 + tl.erf(z * 0.7071067811865476))
    elif MODE == MODE_SILU:
        y = z * tl.sigmoid(z)
    elif MODE == MODE_HSIG:
        y = tl.minimum(tl.maximum((z + 3.0) / 6.0, 0.0), 1.0)
    elif MODE == MODE_HSWISH:
        y = z * tl.minimum(tl.maximum(z + 3.0, 0.0), 6.0) / 6.0
    elif MODE == MODE_SOFTPLUS:
        y = tl.maximum(z, 0.0) + tl.log(1.0 + tl.exp(-tl.abs(z)))
    elif MODE == MODE_CLAMP:
        y = tl.clamp(z, 0.0, 1.0)
    else:
        y = tl.where(z >= 0.0, z, 0.01 * z)

    tl.store(out_ptr + offsets, y, mask=mask)


def bias_activation(x, bias, mode, BLOCK_M=64, BLOCK_N=64):
    '''Method 03: Y = act(X + bias) in one 2-D kernel.'''
    assert x.ndim == 2, "x must be [M, N]"
    assert bias.shape == (x.shape[1],), f"bias must be ({x.shape[1]},)"
    x = x.contiguous()
    bias = bias.contiguous()
    out = torch.empty_like(x, dtype=torch.float32)
    M, N = x.shape
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    m9_bias_activation_kernel[grid](
        x, bias, out, M, N,
        MODE=mode, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
    )
    return out


def test_bias_activation():
    require_gpu("m9 fused bias activation")
    torch.manual_seed(0)
    x = torch.randn(7, 13, device=DEVICE)
    bias = torch.randn(13, device=DEVICE)
    refs = [
        torch.relu(x + bias),
        torch.nn.functional.leaky_relu(x + bias, 0.01),
        torch.sigmoid(x + bias),
        torch.tanh(x + bias),
        torch.nn.functional.gelu(x + bias),
        torch.nn.functional.silu(x + bias),
        torch.nn.functional.hardsigmoid(x + bias),
        torch.nn.functional.hardswish(x + bias),
        torch.nn.functional.softplus(x + bias),
        torch.clamp(x + bias, 0.0, 1.0),
    ]
    for mode, ref in enumerate(refs):
        check(f"bias + {MODE_NAMES[mode]}", bias_activation(x, bias, mode),
              ref, atol=2e-5, rtol=2e-5)

    # Partial tiles: M=7, N=13 with 4x8 blocks.
    check("bias + relu partial tiles",
          bias_activation(x, bias, MODE_RELU, BLOCK_M=4, BLOCK_N=8),
          torch.relu(x + bias))


run_if_gpu(test_bias_activation)

## 12. A gated epilogue: SiLU as a gate

A gated linear unit multiplies a value by a sigmoid gate:

$$
\operatorname{GLU}(a,b)=a\odot\sigma(b).
$$

The gated SiLU (SwiGLU) uses

$$
\operatorname{SwiGLU}(a,b)=a\odot\operatorname{SiLU}(b)
=a\odot b\odot\sigma(b).
$$

It is the activation inside many transformer MLPs. The kernel below takes two
inputs of the same shape and writes one output. It is elementwise, so the mask
and the tile map are the same as before.

**Shape rule.** `a` and `b` must have the same shape (or be broadcast by the
host). The output has that shape. A gate is a Hadamard product, not a matrix
product; the trailing dimension is not contracted.

In [ ]:
@triton.jit
def m9_swiglu_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    a = tl.load(a_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    b = tl.load(b_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    y = a * b * tl.sigmoid(b)
    tl.store(out_ptr + offs, y, mask=mask)


def swiglu(a, b, BLOCK=256):
    a = torch.as_tensor(a).contiguous()
    b = torch.as_tensor(b).contiguous()
    assert a.shape == b.shape, f"same-shape rule: {a.shape} vs {b.shape}"
    out = torch.empty_like(a, dtype=torch.float32)
    m9_swiglu_kernel[(triton.cdiv(a.numel(), BLOCK),)](
        a, b, out, a.numel(), BLOCK=BLOCK
    )
    return out


def test_swiglu():
    require_gpu("m9 SwiGLU")
    torch.manual_seed(0)
    a = torch.randn(7, 13, device=DEVICE)
    b = torch.randn(7, 13, device=DEVICE)
    check("swiglu", swiglu(a, b), a * b * torch.sigmoid(b), atol=1e-6, rtol=1e-6)


run_if_gpu(test_swiglu)

## 13. Softplus with a threshold

The last applied example clamps the output of softplus:

$$
y=\min\left(\operatorname{softplus}(x),h\right).
$$

The threshold is useful for a bounded positive gate. The important detail is the
order: compute the stable softplus first, then clamp. Clamping the input to a
large value before the exponential would change the function and could still
overflow if the threshold is large.

The kernel is a single pass: load, stable softplus, clamp, store. The threshold
is a runtime scalar, so one compiled kernel serves many bounds.

In [ ]:
@triton.jit
def m9_softplus_threshold_kernel(x_ptr, out_ptr, n_elements, hi,
                                 BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    sp = tl.maximum(x, 0.0) + tl.log(1.0 + tl.exp(-tl.abs(x)))
    y = tl.minimum(sp, hi)
    tl.store(out_ptr + offs, y, mask=mask)


def softplus_threshold(x, hi=10.0, BLOCK=256):
    out = torch.empty_like(x, dtype=torch.float32)
    m9_softplus_threshold_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), hi, BLOCK=BLOCK
    )
    return out


def test_softplus_threshold():
    require_gpu("m9 softplus threshold")
    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE) * 20.0
    check("softplus <= 10", softplus_threshold(x, 10.0),
          torch.nn.functional.softplus(x).clamp(max=10.0), atol=1e-5, rtol=1e-5)
    assert softplus_threshold(x, 10.0).max().item() <= 10.0 + 1e-6


run_if_gpu(test_softplus_threshold)

# Capstone — a fused MLP epilogue

The capstone combines everything: a 2-D tile, a bias broadcast, a compile-time
activation, and a stable softplus option. It is the epilogue of a linear layer:

$$
Y_{mn}=\operatorname{act}\left(\sum_k A_{mk}W_{nk}+b_n\right).
$$

The matrix product is assumed to have happened already; the capstone starts from
$X=AW^\top$. The kernel is the same as the fused bias + activation, but the
activation selector includes every mode and the bias is loaded once per column
tile. The test checks all modes at once on an `(M,N)` tile that is not a
multiple of the block sizes.

**Why this is the production shape.** A real epilogue fuses the GEMM accumulator
with the bias and the activation, so the accumulator never round-trips through
HBM. The arithmetic in this capstone is the same; only the source of $X$ changes.

In [ ]:
def mlp_epilogue(x, bias, mode, BLOCK_M=4, BLOCK_N=8):
    '''Capstone: Y = act(X + bias) with a 2-D grid and partial tiles.'''
    return bias_activation(x, bias, mode, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)


def test_capstone():
    require_gpu("m9 capstone")
    torch.manual_seed(0)
    x = torch.randn(9, 17, device=DEVICE)
    bias = torch.randn(17, device=DEVICE)
    refs = [
        torch.relu(x + bias),
        torch.nn.functional.leaky_relu(x + bias, 0.01),
        torch.sigmoid(x + bias),
        torch.tanh(x + bias),
        torch.nn.functional.gelu(x + bias),
        torch.nn.functional.silu(x + bias),
        torch.nn.functional.hardsigmoid(x + bias),
        torch.nn.functional.hardswish(x + bias),
        torch.nn.functional.softplus(x + bias),
        torch.clamp(x + bias, 0.0, 1.0),
    ]
    for mode, ref in enumerate(refs):
        out = mlp_epilogue(x, bias, mode)
        check(f"capstone {MODE_NAMES[mode]}", out, ref, atol=2e-5, rtol=2e-5)
        assert out.shape == x.shape


run_if_gpu(test_capstone)

# Forward and backward

## 14. The derivative table, checked with autograd

The backward of an activation is one elementwise kernel:

$$
dx=g\,f'(x).
$$

The table below is the complete local-derivative set. The code computes each
forward, runs autograd, and compares the result with the manual expression from
Part I. The spine avoids the non-differentiable joins except where the
convention is explicitly tested.

| Activation | $f'(x)$ |
|---|---|
| ReLU | $[x\gt 0]$ |
| Leaky ReLU | $1$ if $x\gt 0$, $\alpha$ otherwise |
| Sigmoid | $\sigma(1-\sigma)$ |
| Tanh | $1-\tanh^2$ |
| GELU | $\Phi(x)+x\phi(x)$ |
| SiLU | $\sigma(1+x(1-\sigma))$ |
| Hard sigmoid | $1/6$ on $(-3,3)$, else $0$ |
| Hard swish | $0$ if $x\lt -3$, $(2x+3)/6$ if $-3\lt x\lt 3$, $1$ if $x\gt 3$ |
| Softplus | $\sigma(x)$ |
| Clamp | $1$ on $(l,u)$, else $0$ |

**What the check proves.** A wrong derivative usually shows up as a constant
factor or a missing term. The GELU row is the one with two terms; the SiLU row is
the one with a product rule. If the maximum error is at rounding level, the
derivation is right. If it is a factor of two, the missing term is visible in the
table.

In [ ]:
def test_backward_table():
    x = torch.tensor([-1.5, -0.5, 0.5, 1.5], requires_grad=True)
    s = torch.sigmoid(x)
    t = torch.tanh(x)
    u = x / math.sqrt(2.0)
    manual = {
        "relu": (x.detach() > 0).float(),
        "sigmoid": (s * (1 - s)).detach(),
        "tanh": (1 - t * t).detach(),
        "gelu": (0.5 * (1 + torch.erf(u))
                 + x.detach() / math.sqrt(2 * math.pi) * torch.exp(-x.detach() ** 2 / 2)).detach(),
        "silu": (s * (1 + x.detach() * (1 - s))).detach(),
        "hardsigmoid": (((x.detach() > -3) & (x.detach() < 3)).float() / 6).detach(),
        "hardswish": torch.where(
            x.detach() < -3, torch.zeros_like(x.detach()),
            torch.where(x.detach() > 3, torch.ones_like(x.detach()),
                        (2 * x.detach() + 3) / 6)).detach(),
        "softplus": s.detach(),
    }
    fns = {
        "relu": torch.relu,
        "sigmoid": torch.sigmoid,
        "tanh": torch.tanh,
        "gelu": torch.nn.functional.gelu,
        "silu": torch.nn.functional.silu,
        "hardsigmoid": torch.nn.functional.hardsigmoid,
        "hardswish": torch.nn.functional.hardswish,
        "softplus": torch.nn.functional.softplus,
    }
    for name, fn in fns.items():
        xi = x.detach().clone().requires_grad_(True)
        fn(xi).sum().backward()
        assert (xi.grad - manual[name]).abs().max().item() < 1e-5, name
    print("backward table: all manual derivatives match autograd")


test_backward_table()

## 15. Why the backward is another elementwise kernel

The forward kernel and the backward kernel have the same structure:

```text
forward :  load x          -> f(x)          -> store y
backward:  load x, load g  -> g * f'(x)     -> store dx
```

They have the same grid, the same mask, and the same traffic pattern (plus one
extra input for `g`). For a fused forward, the backward can either recompute
$f'(x)$ from the saved input or load a saved mask. Recomputing is usually
cheaper in memory and more expensive in arithmetic; for a memory-bound
elementwise kernel, recomputing is the better default.

**A bias in the backward.** If the forward broadcast a bias, the backward sums
the gradient over the broadcast axis:

$$
\frac{\partial L}{\partial b_n}
=
\sum_{m=1}^{M}\frac{\partial L}{\partial Y_{mn}}.
$$

That sum is a reduction, so the bias gradient is **not** an elementwise kernel
of the same shape as the output. It is a reduction over the row axis, which is
the Chapter 14 story and the reason a fused epilogue is often paired with a
small column-reduction kernel for `db`.

In [ ]:
# Check the bias-gradient rule on a tiny case.
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], requires_grad=True)
bias = torch.tensor([0.5, -0.5, 1.0], requires_grad=True)
Y = torch.relu(X + bias)
Y.sum().backward()

assert X.shape == (2, 3) and bias.shape == (3,)
assert X.grad.shape == (2, 3) and bias.grad.shape == (3,)
assert torch.equal(X.grad, torch.ones_like(X))
assert bias.grad.tolist() == [2.0, 2.0, 2.0]
print("bias gradient: the broadcast axis is summed, not copied")

# Pitfalls, checks, and summary

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| `log(1+exp(x))` in `float32` | `inf` for $x\gtrsim88.7$ | stable softplus `max(x,0)+log(1+exp(-|x|))` |
| `exp(x)/(1+exp(x))` | `nan` for large positive $x$ | `tl.sigmoid(x)` or `1/(1+exp(-x))` |
| tanh as a quotient of exponentials | overflow near $\lvert x\rvert\approx44$ | `libdevice.tanh` or `2*sigmoid(2x)-1` |
| GELU tanh approximation tested at `1e-6` | false failure (it is only $\sim10^{-3}$ accurate) | compare to the exact form with a stated tolerance |
| ReLU derivative at zero | convention differs (`>` vs `>=`) | state the convention and test the boundary |
| hard activation boundaries | derivative jumps at $\pm3$ | test the joins explicitly |
| `tl.where` with a dangerous arm | `inf`/`nan` computed before selection | guard the input or use a stable form |
| bias shape `(M,)` for `(M,N)` | not broadcastable | use `(N,)` or reshape to `(M,1)` |
| forgetting the column mask in a 2-D tile | out-of-bounds bias/output access | `row_mask[:,None] & col_mask[None,:]` |
| narrow dtype throughout | overflow/underflow in `exp` or the product | promote to `float32`, cast once at store |
| `tl.exp` on `float16` | type-rule error on some versions | cast to `float32` first |
| clamping the output instead of stabilizing the input | hides overflow and changes the function | stabilize first, clamp only if the function requires it |

## Shape-rule recap

| Situation | Rule | Result |
|---|---|---|
| activation of `x` | elementwise lift | same shape |
| `X (M,N)` + `bias (N,)` | broadcast on the last axis | `(M,N)` |
| `X (M,N)` + `bias (M,1)` | broadcast on the column axis | `(M,N)` |
| `X (M,N)` + `bias (M,)` | last axes $N$ vs $M$ | error unless $N=M$ or one is 1 |
| 2-D tile | `rows<M` and `cols<N`, then outer AND | `(B_M,B_N)` |
| gated SiLU | `a` and `b` same shape | same shape |
| bias gradient | sum over the broadcast row axis | `(N,)` |

## Self-check

1. Compute ReLU, sigmoid, tanh, GELU, SiLU, hard sigmoid, hard swish, and
   softplus at $x=-1,0,1$ to three decimals.
2. Why is $1/(1+e^{-x})$ stable in `float32` while $e^x/(1+e^x)$ is not?
3. Derive the GELU derivative and evaluate it at $x=0$.
4. Derive the SiLU derivative and show it equals $\sigma(1+x(1-\sigma))$.
5. Write the stable softplus and explain why each term is finite.
6. What is the output shape of `(7,13) + (13,)`? Of `(7,13) + (7,)`?
7. Which mask does the 2-D fused kernel use for the bias load? Why not the full
   2-D mask?
8. Why is the bias gradient a reduction rather than an elementwise kernel?
9. Where do hard sigmoid and hard swish have derivative jumps? What convention
   does PyTorch use?
10. Fuse `X @ W.T + b` with GELU and count the saved traffic for
    $M=4096$, $N=4096$, `float32`.

## Summary

- An activation is an elementwise lift; its forward and backward preserve the
  input shape.
- One Triton kernel with a `tl.constexpr` mode covers the whole activation zoo
  without runtime branching.
- Stability is a property of the algebraic form, not of a clamp: use the stable
  sigmoid, tanh, GELU, and softplus forms.
- A fused epilogue adds a bias with the broadcast rule on the last axis and
  applies the activation in registers.
- The backward of a fused activation is another elementwise kernel; the bias
  gradient is a reduction over the broadcast axis.
- Boundary conventions at kinks are part of the API and must be tested.

**Next:** [Chapter 10 — Broadcasting](../../part4_broadcasting/ch10_broadcasting/10_complete.ipynb)

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch08_comparisons/08_complete.ipynb)

# Exercises

1. **Stable sigmoid.** Compare `1/(1+exp(-x))`, `exp(x)/(1+exp(x))`, and
   `tl.sigmoid(x)` on `[-100, 100]`. Which forms produce `nan`, and where?
2. **GELU error.** Plot or print the maximum absolute error of the tanh
   approximation on `[-5, 5]` and on `[-10, 10]`. At which $x$ is it largest?
3. **Boundary gradients.** Feed `[-3, 3]` to hard sigmoid and hard swish and
   compare PyTorch's gradient with the kernel's stated convention.
4. **Leaky slope.** Make `alpha` a runtime scalar instead of a `tl.constexpr`.
   What changes in the compiled kernel, and what stays the same?
5. **Bias broadcast.** Add a per-row bias `(M,1)` to an `(M,N)` tile. Which
   index do you load, and what is the output shape?
6. **SwiGLU.** Derive the backward of $a\odot b\odot\sigma(b)$ with respect to
   both `a` and `b`. Check it with autograd.
7. **Softplus threshold.** Compare `min(softplus(x), h)` with
   `softplus(min(x, h))`. Are they the same function? Which one is finite?
8. **Mixed precision.** Write a `float16` input / `float32` compute version of
   the activation kernel and check the maximum error against PyTorch for GELU
   and softplus.
9. **Fusion traffic.** For $M=N=4096$ and `float32`, compute the bytes saved by
   fusing the bias add into the activation. How many milliseconds does that save
   at 1.5 TB/s?
10. **One selector.** Add `MODE_SOFTPLUS_THRESHOLD` to the activation kernel
    with a runtime `hi`. Why is `hi` not a `tl.constexpr`?

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_naive_reference()
test_backward_table()

print("\nRunning GPU tests (skipped without CUDA/ROCm) ...")
for fn in (
    test_activation_kernel,
    test_stability_forms,
    test_bias_activation,
    test_swiglu,
    test_softplus_threshold,
    test_capstone,
):
    run_if_gpu(fn)